In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.read_csv("../data/2020-Apr-L.csv")

In [11]:
# CONFIGURACIÓN VISUAL
COLORS = {
    "view":     "#A8C4D4",   # azul suave
    "cart":     "#F0A86B",   # naranja medio
    "purchase": "#5B8C6E",   # verde conversión
}

CONVERSION_COLOR_LOW  = "#FDE68A"   # amarillo — baja conversión
CONVERSION_COLOR_HIGH = "#15803D"   # verde oscuro — alta conversión

plt.rcParams.update({
    "font.family":  "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.spines.left":   False,
    "axes.grid":          False,
})

In [12]:
# tasa de conversión: purchase_count / total_events
def conversion_rate(group):
    total    = len(group)
    purchase = (group["event_type"] == "purchase").sum()
    return purchase / total if total > 0 else 0

In [13]:
# PREPARACIÓN DE DATOS — NIVEL 2
def prep_level2(df):
    df = df.copy()
    df["cat_l2"] = df["category_code"].apply(lambda x: ".".join(x.split(".")[:2]))

    # Proporción de cada event_type por cat_l2
    pivot = (
        df.groupby(["cat_l2", "event_type"])
        .size()
        .unstack(fill_value=0)
    )
    # Asegurar las 3 columnas aunque no existan
    for col in ["view", "cart", "purchase"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot = pivot[["view", "cart", "purchase"]]

    # Normalizar a porcentaje
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    # Tasa de conversión por categoría
    conv = df.groupby("cat_l2").apply(conversion_rate).rename("conv_rate")

    # Frecuencia absoluta total
    freq = df.groupby("cat_l2").size().rename("freq")

    # Filtro de ruido: excluir < 0.5 % del total
    total_events = len(df)
    freq_pct = freq / total_events * 100
    mask = freq_pct >= 0.5

    pivot_pct = pivot_pct[mask]
    conv      = conv[mask]
    freq      = freq[mask]

    # Ordenar por frecuencia descendente
    order = freq.sort_values(ascending=False).index
    return pivot_pct.loc[order], conv.loc[order], freq.loc[order]

In [14]:
# PREPARACIÓN DE DATOS — NIVEL 3 (solo apparel.shoes)
def prep_level3_shoes(df):
    df = df.copy()
    mask_l3 = df["category_code"].apply(lambda x: len(x.split(".")) == 3)
    df_shoes = df[mask_l3 & df["category_code"].str.startswith("apparel.shoes")]
    df_shoes = df_shoes.copy()
    df_shoes["cat_l3"] = df_shoes["category_code"]

    pivot = (
        df_shoes.groupby(["cat_l3", "event_type"])
        .size()
        .unstack(fill_value=0)
    )
    for col in ["view", "cart", "purchase"]:
        if col not in pivot.columns:
            pivot[col] = 0
    pivot = pivot[["view", "cart", "purchase"]]
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

    conv = df_shoes.groupby("cat_l3").apply(conversion_rate).rename("conv_rate")
    freq = df_shoes.groupby("cat_l3").size().rename("freq")

    # Filtro ruido: < 0.5 % del total de shoes nivel 3
    total_shoes = len(df_shoes)
    freq_pct = freq / total_shoes * 100
    mask = freq_pct >= 0.5

    pivot_pct = pivot_pct[mask]
    conv      = conv[mask]
    freq      = freq[mask]

    order = freq.sort_values(ascending=False).index
    return pivot_pct.loc[order], conv.loc[order], freq.loc[order]

In [20]:
# FUNCIÓN PRINCIPAL DE GRÁFICO
def plot_stacked(ax, pivot_pct, conv, freq, title, subtitle,
                 show_xlabel=True, freq_total=None):
    categories = pivot_pct.index.tolist()
    x = np.arange(len(categories))
    bar_w = 0.55

    # Barras apiladas
    bottom = np.zeros(len(categories))
    bars_handles = []
    for event in ["view", "cart", "purchase"]:
        vals = pivot_pct[event].values
        bars = ax.bar(x, vals, bar_w,
                      bottom=bottom,
                      color=COLORS[event],
                      label=event.capitalize(),
                      zorder=2)
        bars_handles.append(bars)
        bottom += vals

    # Eje Y izquierdo
    ax.set_ylim(0, 115)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_yticklabels(["0%", "25%", "50%", "75%", "100%"], fontsize=9)
    ax.yaxis.set_tick_params(length=0)

    # Líneas de cuadrícula horizontales suaves
    for y in [25, 50, 75, 100]:
        ax.axhline(y, color="#E5E7EB", linewidth=0.8, zorder=1)

    # Eje Y derecho: Tasa de conversión
    ax2 = ax.twinx()
    conv_vals = conv.values * 100   # en porcentaje
    ax2.plot(x, conv_vals, color="#1E293B", linewidth=1.8,
             marker="o", markersize=5, zorder=5, label="Conv. Rate")
    ax2.set_ylim(0, max(conv_vals.max() * 2, 5))
    ax2.set_ylabel("Tasa de Conversión (%)", fontsize=9, color="#1E293B")
    ax2.yaxis.set_tick_params(length=0, labelsize=8)
    ax2.spines["right"].set_visible(True)
    ax2.spines["right"].set_color("#CBD5E1")

    # Etiquetas de tasa de conversión sobre cada punto
    for xi, cv in zip(x, conv_vals):
        ax2.annotate(f"{cv:.1f}%",
                     xy=(xi, cv),
                     xytext=(0, 7),
                     textcoords="offset points",
                     ha="center", fontsize=7.5,
                     color="#1E293B", fontweight="bold")

    # Etiquetas de frecuencia absoluta bajo barras
    total = freq_total if freq_total else freq.sum()
    for xi, (cat, f) in enumerate(freq.items()):
        pct = f / total * 100
        ax.text(xi, -7, f"{pct:.1f}%",
                ha="center", va="top", fontsize=7.5,
                color="#6B7280")

    # Ejes X
    labels = [c.replace("apparel.", "").replace("accessories.", "acc.")
               .replace("sport.", "sport.") for c in categories]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9, rotation=30, ha="right")
    ax.tick_params(axis="x", length=0)
    if not show_xlabel:
        ax.set_xticklabels([])

    # Títulos
    titulo_completo = f"{title}"
    ax.set_title(titulo_completo, fontsize=12, fontweight="bold",
                 pad=22, loc="left", color="#0F172A")
    ax.text(0.0, 1.03, subtitle, transform=ax.transAxes,
            fontsize=8.5, color="#64748B", va="bottom")

    ax.spines["bottom"].set_color("#E5E7EB")

    return ax, ax2

In [22]:
# RENDER COMPLETO
def render_graficos(df):
    pct_l2, conv_l2, freq_l2 = prep_level2(df)
    pct_l3, conv_l3, freq_l3 = prep_level3_shoes(df)

    fig, (ax1, ax2) = plt.subplots(
        2, 1,
        figsize=(13, 11),
        gridspec_kw={"hspace": 0.55}
    )

    fig.patch.set_facecolor("#F8FAFC")
    ax1.set_facecolor("#F8FAFC")
    ax2.set_facecolor("#F8FAFC")

    # Gráfico 1: Nivel 2
    plot_stacked(
        ax1, pct_l2, conv_l2, freq_l2,
        title="Distribución por Categoría (Nivel 2)",
        subtitle="Composición de eventos por subcategoría · % debajo = peso relativo sobre total filtrado"
    )

    # Gráfico 2: Zoom apparel.shoes Nivel 3
    plot_stacked(
        ax2, pct_l3, conv_l3, freq_l3,
        title="Zoom: apparel.shoes (Nivel 3)",
        subtitle="Descomposición interna de apparel.shoes · % debajo = peso relativo dentro del grupo shoes"
    )

    # Leyenda global
    legend_patches = [
        mpatches.Patch(color=COLORS["view"],     label="View"),
        mpatches.Patch(color=COLORS["cart"],     label="Cart"),
        mpatches.Patch(color=COLORS["purchase"], label="Purchase"),
    ]
    conv_line = plt.Line2D([0], [0], color="#1E293B", linewidth=1.8,
                           marker="o", markersize=5, label="Conv. Rate")
    legend_patches.append(conv_line)

    fig.legend(
        handles=legend_patches,
        loc="lower center",
        ncol=4,
        fontsize=9,
        frameon=False,
        bbox_to_anchor=(0.5, -0.01)
    )

    # Nota de filtro de ruido
    fig.text(0.01, -0.03,
             "* Subcategorías con <0.5% del total excluidas (filtro de ruido estadístico)",
             fontsize=7.5, color="#9CA3AF", style="italic")

    plt.savefig("../eda_outputs/02_dist_categories.png",
                dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    plt.close(fig)
    print("¡Gráfico 02_dist_categories.png guardado exitosamente!")

In [23]:
render_graficos(df);

¡Gráfico 02_dist_categories.png guardado exitosamente!
